In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/mdrashidshahariar/notebook-02-dataset-generation/__results__.html
/kaggle/input/notebooks/mdrashidshahariar/notebook-02-dataset-generation/__notebook__.ipynb
/kaggle/input/notebooks/mdrashidshahariar/notebook-02-dataset-generation/__output__.json
/kaggle/input/notebooks/mdrashidshahariar/notebook-02-dataset-generation/custom.css
/kaggle/input/notebooks/mdrashidshahariar/notebook-02-dataset-generation/SCD_PreVF/5_min/47.npy
/kaggle/input/notebooks/mdrashidshahariar/notebook-02-dataset-generation/SCD_PreVF/5_min/45.npy
/kaggle/input/notebooks/mdrashidshahariar/notebook-02-dataset-generation/SCD_PreVF/5_min/43.npy
/kaggle/input/notebooks/mdrashidshahariar/notebook-02-dataset-generation/SCD_PreVF/5_min/50.npy
/kaggle/input/notebooks/mdrashidshahariar/notebook-02-dataset-generation/SCD_PreVF/5_min/30.npy
/kaggle/input/notebooks/mdrashidshahariar/notebook-02-dataset-generation/SCD_PreVF/5_min/44.npy
/kaggle/input/notebooks/mdrashidshahariar/notebook-02-dataset-generat

In [2]:
pip install wfdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 3.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import gc
import random
import numpy as np
import pandas as pd
import wfdb
from tqdm import tqdm

In [4]:
import os

# This will print the exact folder name Kaggle generated for your notebook output
print(os.listdir("../input"))

# To access your folders (like 5_min, 10_min, etc.), your path will look like this:
# PATH = "../input/YOUR_NOTEBOOK_2_URL_SLUG/SCD_PreVF"

['notebooks', 'datasets']


In [5]:
# ============================================================
# Notebook 2 Output
# ============================================================

PREVF_ROOT = "/kaggle/input/notebooks/mdrashidshahariar/notebook-02-dataset-generation/SCD_PreVF"

# ============================================================
# Original Dataset
# ============================================================

ROOT = "/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter"

NSR_PATH = os.path.join(ROOT, "nsrdb_250hz")

# ============================================================
# Output
# ============================================================

OUTPUT_ROOT = "/kaggle/working/Balanced_Dataset"

In [6]:
FS = 250

WINDOW_DURATION = 2      # minutes

SEGMENT_DURATION = 2     # seconds

WINDOW_SAMPLES = WINDOW_DURATION * 60 * FS

SEGMENT_SAMPLES = SEGMENT_DURATION * FS

prediction_windows = [30,25,20,15,10,5]

random.seed(42)

np.random.seed(42)

In [7]:
for minute in prediction_windows:

    folder = os.path.join(
        PREVF_ROOT,
        f"{minute}_min"
    )

    files = sorted(os.listdir(folder))

    print(f"{minute} min")

    print("Patients :", len(files))

    print(files[:5])

    print("-"*40)

30 min
Patients : 20
['30.npy', '31.npy', '32.npy', '33.npy', '34.npy']
----------------------------------------
25 min
Patients : 20
['30.npy', '31.npy', '32.npy', '33.npy', '34.npy']
----------------------------------------
20 min
Patients : 20
['30.npy', '31.npy', '32.npy', '33.npy', '34.npy']
----------------------------------------
15 min
Patients : 20
['30.npy', '31.npy', '32.npy', '33.npy', '34.npy']
----------------------------------------
10 min
Patients : 20
['30.npy', '31.npy', '32.npy', '33.npy', '34.npy']
----------------------------------------
5 min
Patients : 20
['30.npy', '31.npy', '32.npy', '33.npy', '34.npy']
----------------------------------------


In [8]:
sample = np.load(

    os.path.join(

        PREVF_ROOT,

        "30_min",

        "30.npy"

    )

)

print(sample.shape)

(450000, 2)


In [9]:
def extract_paper_window(ecg):

    """
    Extracts the FIRST 2 minutes
    from Notebook 2 windows.

    Example

    30 min folder:

    VF-30 ---------- VF

    returns

    VF-30 ---- VF-28
    """

    return ecg[:WINDOW_SAMPLES]

In [10]:
def split_segments(ecg):

    segments = []

    total = len(ecg)

    for start in range(

        0,

        total,

        SEGMENT_SAMPLES

    ):

        end = start + SEGMENT_SAMPLES

        if end <= total:

            segments.append(

                ecg[start:end]

            )

    return segments

In [11]:
window = extract_paper_window(sample)

segments = split_segments(window)

print(window.shape)

print(len(segments))

print(segments[0].shape)

(30000, 2)
60
(500, 2)


In [12]:
# ============================================================
# Create Output Folders
# ============================================================

for minute in prediction_windows:

    os.makedirs(
        os.path.join(
            OUTPUT_ROOT,
            f"{minute}_min",
            "Positive"
        ),
        exist_ok=True
    )

    os.makedirs(
        os.path.join(
            OUTPUT_ROOT,
            f"{minute}_min",
            "Negative"
        ),
        exist_ok=True
    )

print("Folders created successfully.")

Folders created successfully.


In [13]:
# ============================================================
# Generate Positive ECG Segments
# ============================================================

for minute in prediction_windows:

    print(f"\nGenerating Positive Dataset ({minute} min)")

    folder = os.path.join(
        PREVF_ROOT,
        f"{minute}_min"
    )

    save_folder = os.path.join(
        OUTPUT_ROOT,
        f"{minute}_min",
        "Positive"
    )

    segment_count = 0

    files = sorted(os.listdir(folder))

    for file in files:

        ecg = np.load(
            os.path.join(folder, file)
        )

        # Paper uses first 2 minutes
        ecg = extract_paper_window(ecg)

        segments = split_segments(ecg)

        patient_id = file.replace(".npy", "")

        for idx, segment in enumerate(segments):

            filename = f"{patient_id}_{idx+1}.npy"

            np.save(
                os.path.join(save_folder, filename),
                segment
            )

            segment_count += 1

    print("Total Positive Segments:", segment_count)


Generating Positive Dataset (30 min)
Total Positive Segments: 1200

Generating Positive Dataset (25 min)
Total Positive Segments: 1200

Generating Positive Dataset (20 min)
Total Positive Segments: 1200

Generating Positive Dataset (15 min)
Total Positive Segments: 1200

Generating Positive Dataset (10 min)
Total Positive Segments: 1200

Generating Positive Dataset (5 min)
Total Positive Segments: 1200


In [14]:
# ============================================================
# Verify Positive Dataset
# ============================================================

for minute in prediction_windows:

    folder = os.path.join(
        OUTPUT_ROOT,
        f"{minute}_min",
        "Positive"
    )

    files = sorted(os.listdir(folder))

    print(f"{minute} min")

    print("Segments :", len(files))

    print("First Files :", files[:5])

    print("-"*40)

30 min
Segments : 1200
First Files : ['30_1.npy', '30_10.npy', '30_11.npy', '30_12.npy', '30_13.npy']
----------------------------------------
25 min
Segments : 1200
First Files : ['30_1.npy', '30_10.npy', '30_11.npy', '30_12.npy', '30_13.npy']
----------------------------------------
20 min
Segments : 1200
First Files : ['30_1.npy', '30_10.npy', '30_11.npy', '30_12.npy', '30_13.npy']
----------------------------------------
15 min
Segments : 1200
First Files : ['30_1.npy', '30_10.npy', '30_11.npy', '30_12.npy', '30_13.npy']
----------------------------------------
10 min
Segments : 1200
First Files : ['30_1.npy', '30_10.npy', '30_11.npy', '30_12.npy', '30_13.npy']
----------------------------------------
5 min
Segments : 1200
First Files : ['30_1.npy', '30_10.npy', '30_11.npy', '30_12.npy', '30_13.npy']
----------------------------------------


In [15]:
sample = np.load(

    os.path.join(

        OUTPUT_ROOT,

        "30_min",

        "Positive",

        "30_1.npy"

    )

)

print(sample.shape)

(500, 2)


In [16]:
"""Reproducibility Note

Reproducibility Note

The original paper reports a balanced dataset consisting of 1200 normal ECG segments, but it does not describe how these segments were selected from the 18 Normal Sinus Rhythm (NSR) recordings. Specifically, the paper does not specify the starting positions of the 2-minute windows, whether the windows were selected randomly or systematically, whether overlapping windows were permitted, or how multiple windows were distributed across subjects.

To reproduce the reported dataset size while maintaining subject representation, this implementation extracts one random 2-minute window from each of the 18 NSR subjects and two additional non-overlapping 2-minute windows from two randomly selected subjects, resulting in 20 two-minute windows. Each window is then divided into sixty non-overlapping 2-second ECG segments, yielding 1200 normal ECG segments, consistent with the dataset size reported in the original study.

This implementation choice is explicitly documented because the original publication does not provide sufficient methodological detail for exact reproduction."""

'Reproducibility Note\n\nReproducibility Note\n\nThe original paper reports a balanced dataset consisting of 1200 normal ECG segments, but it does not describe how these segments were selected from the 18 Normal Sinus Rhythm (NSR) recordings. Specifically, the paper does not specify the starting positions of the 2-minute windows, whether the windows were selected randomly or systematically, whether overlapping windows were permitted, or how multiple windows were distributed across subjects.\n\nTo reproduce the reported dataset size while maintaining subject representation, this implementation extracts one random 2-minute window from each of the 18 NSR subjects and two additional non-overlapping 2-minute windows from two randomly selected subjects, resulting in 20 two-minute windows. Each window is then divided into sixty non-overlapping 2-second ECG segments, yielding 1200 normal ECG segments, consistent with the dataset size reported in the original study.\n\nThis implementation choic

In [17]:
# ============================================================
# Load NSR Record IDs
# ============================================================

nsr_records = sorted([

    f.replace(".hea", "")

    for f in os.listdir(NSR_PATH)

    if f.endswith(".hea")

])

print("Total NSR Subjects:", len(nsr_records))
print(nsr_records)

Total NSR Subjects: 18
['16265', '16272', '16273', '16420', '16483', '16539', '16773', '16786', '16795', '17052', '17453', '18177', '18184', '19088', '19090', '19093', '19140', '19830']


In [18]:
# ============================================================
# Random 2-Minute Window Extraction
# ============================================================

def extract_random_window(record_path,
                          window_minutes=2,
                          fs=250):

    record = wfdb.rdrecord(record_path)

    signal = record.p_signal

    window_samples = window_minutes * 60 * fs

    total_samples = len(signal)

    max_start = total_samples - window_samples

    start = np.random.randint(0, max_start)

    end = start + window_samples

    return signal[start:end], start

In [19]:
record = nsr_records[0]

window, start = extract_random_window(
    os.path.join(NSR_PATH, record)
)

print("Record :", record)
print("Shape  :", window.shape)
print("Start Sample :", start)

Record : 16265
Shape  : (30000, 2)
Start Sample : 16094478


In [20]:
# ============================================================
# Generate 20 Normal Windows
# ============================================================

normal_windows = []

used_starts = {}

# ------------------------------------------------------------
# First pass:
# One random window from each of the 18 NSR subjects
# ------------------------------------------------------------

for record in nsr_records:

    record_path = os.path.join(NSR_PATH, record)

    window, start = extract_random_window(record_path)

    normal_windows.append({
        "Record": record,
        "Window": window,
        "Start": start
    })

    used_starts.setdefault(record, []).append(start)

print(f"Initial windows: {len(normal_windows)}")


# ------------------------------------------------------------
# Second pass:
# Add two extra non-overlapping windows
# ------------------------------------------------------------

extra_subjects = np.random.choice(
    nsr_records,
    size=2,
    replace=False
)

for record in extra_subjects:

    record_path = os.path.join(NSR_PATH, record)

    while True:

        window, start = extract_random_window(record_path)

        overlap = False

        for previous_start in used_starts[record]:

            if abs(start - previous_start) < WINDOW_SAMPLES:
                overlap = True
                break

        if not overlap:

            normal_windows.append({
                "Record": record,
                "Window": window,
                "Start": start
            })

            used_starts[record].append(start)

            break

print(f"Final windows: {len(normal_windows)}")

Initial windows: 18
Final windows: 20


In [21]:
# ============================================================
# Verify Generated Normal Windows
# ============================================================

print("Total Windows :", len(normal_windows))

print()

for item in normal_windows[:5]:

    print(
        f"Record {item['Record']} | "
        f"Shape {item['Window'].shape} | "
        f"Start {item['Start']}"
    )

Total Windows : 20

Record 16265 | Shape (30000, 2) | Start 21081788
Record 16272 | Shape (30000, 2) | Start 13315092
Record 16273 | Shape (30000, 2) | Start 2234489
Record 16420 | Shape (30000, 2) | Start 14586186
Record 16483 | Shape (30000, 2) | Start 9628519


In [22]:
# ============================================================
# Generate Negative ECG Segments
# ============================================================

for minute in prediction_windows:

    print(f"\nGenerating Negative Dataset ({minute} min)")

    save_folder = os.path.join(
        OUTPUT_ROOT,
        f"{minute}_min",
        "Negative"
    )

    segment_count = 0

    for window_idx, item in enumerate(normal_windows):

        patient_id = item["Record"]

        ecg = item["Window"]

        segments = split_segments(ecg)

        for seg_idx, segment in enumerate(segments):

            filename = (
                f"{patient_id}"
                f"_W{window_idx+1}"
                f"_S{seg_idx+1}.npy"
            )

            np.save(
                os.path.join(save_folder, filename),
                segment
            )

            segment_count += 1

    print("Total Negative Segments:", segment_count)


Generating Negative Dataset (30 min)
Total Negative Segments: 1200

Generating Negative Dataset (25 min)
Total Negative Segments: 1200

Generating Negative Dataset (20 min)
Total Negative Segments: 1200

Generating Negative Dataset (15 min)
Total Negative Segments: 1200

Generating Negative Dataset (10 min)
Total Negative Segments: 1200

Generating Negative Dataset (5 min)
Total Negative Segments: 1200


In [23]:
# ============================================================
# Verify Negative Dataset
# ============================================================

for minute in prediction_windows:

    folder = os.path.join(
        OUTPUT_ROOT,
        f"{minute}_min",
        "Negative"
    )

    files = sorted(
        f for f in os.listdir(folder)
        if f.endswith(".npy")
    )

    print(f"\n{minute} min")

    print("Segments :", len(files))

    print("First Files :", files[:5])

    print("-"*40)


30 min
Segments : 1200
First Files : ['16265_W1_S1.npy', '16265_W1_S10.npy', '16265_W1_S11.npy', '16265_W1_S12.npy', '16265_W1_S13.npy']
----------------------------------------

25 min
Segments : 1200
First Files : ['16265_W1_S1.npy', '16265_W1_S10.npy', '16265_W1_S11.npy', '16265_W1_S12.npy', '16265_W1_S13.npy']
----------------------------------------

20 min
Segments : 1200
First Files : ['16265_W1_S1.npy', '16265_W1_S10.npy', '16265_W1_S11.npy', '16265_W1_S12.npy', '16265_W1_S13.npy']
----------------------------------------

15 min
Segments : 1200
First Files : ['16265_W1_S1.npy', '16265_W1_S10.npy', '16265_W1_S11.npy', '16265_W1_S12.npy', '16265_W1_S13.npy']
----------------------------------------

10 min
Segments : 1200
First Files : ['16265_W1_S1.npy', '16265_W1_S10.npy', '16265_W1_S11.npy', '16265_W1_S12.npy', '16265_W1_S13.npy']
----------------------------------------

5 min
Segments : 1200
First Files : ['16265_W1_S1.npy', '16265_W1_S10.npy', '16265_W1_S11.npy', '16265_W

In [24]:
# ============================================================
# Final Dataset Verification
# ============================================================

summary = []

for minute in prediction_windows:

    positive_folder = os.path.join(
        OUTPUT_ROOT,
        f"{minute}_min",
        "Positive"
    )

    negative_folder = os.path.join(
        OUTPUT_ROOT,
        f"{minute}_min",
        "Negative"
    )

    positive = len([
        f for f in os.listdir(positive_folder)
        if f.endswith(".npy")
    ])

    negative = len([
        f for f in os.listdir(negative_folder)
        if f.endswith(".npy")
    ])

    total = positive + negative

    summary.append({
        "Prediction_Minutes": minute,
        "Positive": positive,
        "Negative": negative,
        "Total": total
    })

summary_df = pd.DataFrame(summary)

summary_df

,Prediction_Minutes,Positive,Negative,Total
0,30,1200,1200,2400
1,25,1200,1200,2400
2,20,1200,1200,2400
3,15,1200,1200,2400
4,10,1200,1200,2400
5,5,1200,1200,2400


In [25]:
print("Total normal windows:", len(normal_windows))

Total normal windows: 20


In [26]:
# ============================================================
# Verify Saved ECG Segment
# ============================================================

sample = np.load(
    os.path.join(
        OUTPUT_ROOT,
        "30_min",
        "Positive",
        os.listdir(
            os.path.join(
                OUTPUT_ROOT,
                "30_min",
                "Positive"
            )
        )[0]
    )
)

print("Positive Segment Shape :", sample.shape)

sample = np.load(
    os.path.join(
        OUTPUT_ROOT,
        "30_min",
        "Negative",
        os.listdir(
            os.path.join(
                OUTPUT_ROOT,
                "30_min",
                "Negative"
            )
        )[0]
    )
)

print("Negative Segment Shape :", sample.shape)

Positive Segment Shape : (500, 2)
Negative Segment Shape : (500, 2)


In [27]:
# ============================================================
# Dataset Summary
# ============================================================

summary = []

for minute in prediction_windows:

    positive = len([
        f for f in os.listdir(
            os.path.join(
                OUTPUT_ROOT,
                f"{minute}_min",
                "Positive"
            )
        )
        if f.endswith(".npy")
    ])

    negative = len([
        f for f in os.listdir(
            os.path.join(
                OUTPUT_ROOT,
                f"{minute}_min",
                "Negative"
            )
        )
        if f.endswith(".npy")
    ])

    summary.append({

        "Prediction Horizon (min)": minute,

        "Positive Segments": positive,

        "Negative Segments": negative,

        "Total Segments": positive + negative

    })

summary_df = pd.DataFrame(summary)

summary_df

,Prediction Horizon (min),Positive Segments,Negative Segments,Total Segments
0,30,1200,1200,2400
1,25,1200,1200,2400
2,20,1200,1200,2400
3,15,1200,1200,2400
4,10,1200,1200,2400
5,5,1200,1200,2400


In [28]:
# ============================================================
# Save Dataset Summary
# ============================================================

summary_df.to_csv(
    os.path.join(
        OUTPUT_ROOT,
        "dataset_summary.csv"
    ),
    index=False
)

print("Summary saved successfully.")

Summary saved successfully.


In [29]:
print("="*60)
print("NOTEBOOK 3 COMPLETED")
print("="*60)

print(f"Prediction Horizons : {len(prediction_windows)}")

print(f"Positive Segments per Horizon : 1200")

print(f"Negative Segments per Horizon : 1200")

print(f"Total Segments per Horizon : 2400")

print()

print("Segment Shape : (500, 2)")

print("Sampling Frequency : 250 Hz")

print("Segment Duration : 2 seconds")

print("="*60)

NOTEBOOK 3 COMPLETED
Prediction Horizons : 6
Positive Segments per Horizon : 1200
Negative Segments per Horizon : 1200
Total Segments per Horizon : 2400

Segment Shape : (500, 2)
Sampling Frequency : 250 Hz
Segment Duration : 2 seconds
